# Lab 06 · Classification

Practise training classifiers and use their mistakes to understand how they perform.

Fill each `____` using the comments, then run the cells in order.


## Image Values and Labels


In [ ]:
# A tiny black-and-white parcel sticker is stored row by row as nine brightness values.
import ____ as np                          # NumPy arrays.
import matplotlib.pyplot as ____             # Use the alias plt for plotting.
sticker = np.array([0, 255, 0, 255, 255, 255, 0, 255, ____]) # Last pixel is black (0).
sticker_image = sticker.____(3, 3)         # Recover its three rows and three columns.
plt.figure(figsize=____)                   # A new figure, four inches on each side.
plt.imshow(____, cmap="gray", vmin=0, vmax=255)
plt.title(____)
plt.____()                                  # Display the figure.


In [ ]:
# Separately, learn which of three gift-bag sizes was used for each card.
# Each row is one invented card; side lengths are in centimetres.
import ____ as pd                         # pandas tables.
cards = pd.____()
long_cm = [9,10,11,12,13,14,12,13,14,____]     # Small bags; last long side: 15 cm.
long_cm += [13,14,15,16,17,18,16,17,18,____]  # Medium bags; last long side: 19 cm.
long_cm += [18,19,20,21,22,23,24,25,26,____]  # Large bags; last long side: 27 cm.
short_cm = [6,7,8,6,7,8,9,9,10,____]          # Small bags; last short side: 8 cm.
short_cm += [10,11,9,10,11,12,13,12,14,____] # Medium bags; last short side: 13 cm.
short_cm += [14,15,13,16,15,17,16,18,17,____] # Large bags; last short side: 19 cm.
cards["long_cm"] = ____                 # Add the complete long-side list.
cards["short_cm"] = ____               # Add the complete short-side list.
cards["bag"] = ["small"] * 10 + ["medium"] * 10 + [____] * 10 # Final ten used large bags.
X = cards[["long_cm", ____]]           # The two measured inputs.
y = cards[____]                            # The recorded bag size to predict.
print(y.____())                     # Count the examples of each size.


## Training, Validation, and Test Sets


In [ ]:
# Keep 60% for learning, 20% for choosing a model, and 20% for the final check.
from sklearn.model_selection import ____
X_train, X_other, y_train, y_other = train_test_split(X, y, test_size=____, random_state=17, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_other, y_other, test_size=____, random_state=17, stratify=y_other)
print(len(X_train), len(X_val), ____)   # Report the three row counts.
# Splitting inputs and labels in the same call keeps the correct pairs together.


## Binary Classification and Cross-validation


In [ ]:
# First predict only whether a card needs the large bag.
from sklearn.tree import ____ # Decision tree classifier class.
y_large = y_train ____ "large"
binary_tree = DecisionTreeClassifier(max_leaf_nodes=3, random_state=____) # Reuse seed 17.
binary_tree.____(X_train, y_large)
# Predict bags for cards measuring (13, 9) and (26, 19) cm.
new_cards = pd.DataFrame({"long_cm": [13, 26], "short_cm": [9, ____]})
print(binary_tree.____(new_cards))         # True means the model selected the large bag.


In [ ]:
# Check the binary tree on held-out folds within the training records.
from sklearn.model_selection import ____, cross_val_predict
fold_accuracy = cross_val_score(binary_tree, X_train, y_large, cv=____, scoring="accuracy") # Three folds.
print(____)                         # One accuracy for each fold.
print("Mean accuracy:", fold_accuracy.____())
# Scores here describe only these small invented records.


## Binary Errors, Precision, and Recall


In [ ]:
# Give every training card a prediction from a tree that did not train on that card.
from sklearn.metrics import ____, accuracy_score
large_prediction = cross_val_predict(binary_tree, X_train, ____, cv=3)
binary_matrix = confusion_matrix(y_large, ____, labels=[False, True])
print(____)
# Rows are actual [not large, large]; columns are predicted [not large, large].


In [ ]:
# Precision: how often a large-bag prediction was right. Recall: how many large bags were found.
tp = binary_matrix[____, 1]                    # Actual large and predicted large.
fp = binary_matrix[____, 1]                    # Actual not large but predicted large.
fn = binary_matrix[1, ____]                    # Actual large but predicted not large.
precision = tp / (tp ____ fp)                  # These data have positive denominators.
recall = tp / (tp ____ fn)
print("Precision:", precision, "Recall:", ____)


## Multiclass Classification


In [ ]:
# A decision tree can learn the three bag names directly.
size_tree = DecisionTreeClassifier(max_leaf_nodes=3, random_state=____) # Reuse seed 17.
size_tree.____(X_train, y_train)
print(size_tree.____)                   # The class names known to this tree.
size_prediction = cross_val_predict(size_tree, X_train, ____, cv=3)
print("Training-fold accuracy:", accuracy_score(y_train, ____))


In [ ]:
# Compare each actual bag size with the predicted size in this explicit order.
bag_order = ["small", "medium", ____]
size_matrix = confusion_matrix(y_train, size_prediction, labels=____)
row_totals = size_matrix.sum(axis=____, keepdims=True)
size_rates = size_matrix ____ row_totals        # Fractions within each actual bag size.
print(____)
print(____)


In [ ]:
# Hide correct predictions to focus the picture on mistakes.
error_rates = size_rates.____()
np.fill_diagonal(error_rates, ____)
plt.figure(figsize=____)                  # A new figure, five by four inches.
plt.imshow(____, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(3), ____)
plt.yticks(range(3), ____)
plt.xlabel(____)
plt.ylabel(____)
plt.colorbar(label=____)
plt.____()                                 # Display the error picture.
# A pale diagonal here was set to zero; it is not evidence of zero correct predictions.


In [ ]:
# Inspect the card measurements for mistaken predictions.
review = X_train.____()
review["actual_bag"] = ____
review["predicted_bag"] = ____
mistakes = review[review["actual_bag"] ____ review["predicted_bag"]]
print(____)


## Hyperparameter Selection


In [ ]:
# Compare leaf limits 2, 3, and 5 using validation accuracy; test records stay untouched.
validation_scores = ____                     # Leaf limit -> validation accuracy.
for leaf_limit in [2, 3, ____]:
    candidate = DecisionTreeClassifier(max_leaf_nodes=____, random_state=17)
    candidate.fit(X_train, ____)
    guessed_bags = candidate.____(X_val)
    validation_scores[leaf_limit] = accuracy_score(y_val, ____)
best_limit = max(____, key=validation_scores.get)
print(validation_scores, "Chosen leaf limit:", ____)
# max keeps the first inserted key on a tie; ascending limits therefore prefer the smaller tree.


In [ ]:
# Use the selected limit for one final evaluation on the reserved test cards.
final_tree = DecisionTreeClassifier(max_leaf_nodes=____, random_state=17)
final_tree.fit(X_train, ____)
test_guesses = final_tree.____(X_test)
print("Test accuracy:", accuracy_score(y_test, ____))
print(confusion_matrix(y_test, test_guesses, labels=____))
# These six test cards cannot establish performance on real shopping orders.
